# Transformer-Based Text Models for Endodontic Treatment Outcome Prediction

This notebook contains all text-based transformer experiments for predicting endodontic treatment outcomes from clinical text. Five model configurations are compared.

## 1. Setup & Configuration

In [ ]:
import os, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from datasets import load_dataset, Value
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# ── Global config ────────────────────────────────────────────────────────
TEST_SIZE    = 0.20
VAL_SIZE     = 0.10
RANDOM_STATE = 42
DATA_DIR     = "data"
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 2. Data Preparation & Train/Val/Test Split

The raw file is not included in this repository (patient data).
Expected columns:
- `prognosis_english` — translated free-text prognosis note
- `prognosis_danish` (or equivalent) — original untranslated Danish prognosis note
- `is_effective_romexis` — binary outcome label (0/1)

Adjust `TEXT_COL_DANISH` and `ALL_TEXT_COLS` to match your actual column names.

In [ ]:
TEXT_COL_EN     = "prognosis_english"    # translated prognosis note
TEXT_COL_DANISH = "prognosis_danish"     # original Danish text — update if column name differs
LABEL_COL       = "is_effective_romexis"

# Columns concatenated for the "prognosis + signs & symptoms" experiments.
# Update to match the exact column names in your dataset.
ALL_TEXT_COLS   = ["prognosis_english"]  # e.g. ["prognosis_english", "signs_symptoms_col"]

In [ ]:
path = os.path.join(DATA_DIR, "Erda, final, anonymized,for LLM and multimodal.csv")
df   = pd.read_csv(path)

# ── Clean ────────────────────────────────────────────────────────────────
df = df.dropna(subset=[TEXT_COL_EN, LABEL_COL])
df = df[df[TEXT_COL_EN].str.strip().ne("")]

# Build combined column: prognosis note + signs & symptoms (space-separated)
df["all_texts"] = df[ALL_TEXT_COLS].fillna("").astype(str).agg(" ".join, axis=1).str.strip()

X = df[TEXT_COL_EN]
y = df[LABEL_COL]

# Stratified 70 / 10 / 20 split — indices reused for all experiments
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
val_size_rel = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=val_size_rel, stratify=y_trainval, random_state=RANDOM_STATE
)

# ── Persist splits (text columns + labels) ──────────────────────────────
os.makedirs(DATA_DIR, exist_ok=True)
idx_train, idx_val, idx_test = X_train.index, X_val.index, X_test.index

for split, idx in [("train", idx_train), ("val", idx_val), ("test", idx_test)]:
    df.loc[idx, [TEXT_COL_EN, TEXT_COL_DANISH, "all_texts"]].to_csv(
        f"{DATA_DIR}/texts_{split}.csv", index=False
    )
    df.loc[idx, [LABEL_COL]].rename(columns={LABEL_COL: "labels"}).to_csv(
        f"{DATA_DIR}/labels_{split}.csv", index=False
    )

print(f"Split sizes → train: {len(idx_train)}, val: {len(idx_val)}, test: {len(idx_test)}")

## 3. Dataset Loading & Tokenisation

A single helper loads and tokenises any text column for any checkpoint.
All five experiments call this function with different arguments.

In [ ]:
def build_tokenized_dataset(model_ckpt: str, text_col: str, max_length: int = 128):
    """
    Load the pre-split CSVs, attach labels, and tokenise with `model_ckpt`.

    Args:
        model_ckpt:  HuggingFace model identifier.
        text_col:    Which text column to use (e.g. 'prognosis_english',
                     'prognosis_danish', 'all_texts').
        max_length:  Tokenisation truncation length.

    Returns:
        tokenized_dataset, raw_dataset, tokenizer
    """
    raw = load_dataset("csv", data_files={
        "train":      f"{DATA_DIR}/texts_train.csv",
        "validation": f"{DATA_DIR}/texts_val.csv",
        "test":       f"{DATA_DIR}/texts_test.csv",
    })
    label_ds = load_dataset("csv", data_files={
        "train":      f"{DATA_DIR}/labels_train.csv",
        "validation": f"{DATA_DIR}/labels_val.csv",
        "test":       f"{DATA_DIR}/labels_test.csv",
    })

    for split in raw:
        assert len(raw[split]) == len(label_ds[split]), f"Row mismatch in {split}"
        raw[split] = raw[split].add_column("labels", label_ds[split]["labels"])
    raw = raw.cast_column("labels", Value("int64"))

    tok = AutoTokenizer.from_pretrained(model_ckpt)

    def preprocess(examples):
        texts = [str(t) if t is not None else "" for t in examples[text_col]]
        return tok(texts, padding="max_length", truncation=True, max_length=max_length)

    cols_to_remove = [c for c in raw["train"].column_names if c != "labels"]
    tokenized = raw.map(preprocess, batched=True, remove_columns=cols_to_remove)
    tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return tokenized, raw, tok

## 4. Shared Utilities

Defined once and reused by all five experiments.

In [ ]:
def compute_metrics(eval_pred):
    """Per-class P/R/F1, macro-F1, ROC-AUC, PR-AUC (threshold=0.5 during training only)."""
    logits, y = eval_pred
    logits = logits - logits.max(axis=1, keepdims=True)
    probs  = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
    y_prob = probs[:, 1]
    y_hat  = (y_prob >= 0.5).astype(int)

    prec, rec, f1, _ = precision_recall_fscore_support(y, y_hat, average=None, labels=[0, 1])
    macro_f1 = precision_recall_fscore_support(y, y_hat, average="macro")[2]
    return {
        "precision_0": prec[0], "recall_0": rec[0], "f1_0": f1[0],
        "precision_1": prec[1], "recall_1": rec[1], "f1_1": f1[1],
        "macro_f1":    macro_f1,
        "roc_auc":     roc_auc_score(y, y_prob),
        "pr_auc":      average_precision_score(y, y_prob),
    }

In [ ]:
class FocalLoss(nn.Module):
    """
    Multi-class focal loss (cross-entropy based).

    Args:
        alpha: optional per-class weight tensor of shape [num_classes]
        gamma: focusing parameter (default 2.0)
    """
    def __init__(self, alpha=None, gamma: float = 2.0, reduction: str = "mean"):
        super().__init__()
        self.alpha     = alpha
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets = targets.long()
        weight  = self.alpha.to(logits.device) if isinstance(self.alpha, torch.Tensor) else self.alpha
        ce      = F.cross_entropy(logits, targets, weight=weight, reduction="none")
        pt      = torch.exp(-ce)
        loss    = ((1 - pt) ** self.gamma) * ce
        return loss.mean() if self.reduction == "mean" else loss.sum()


class FocalTrainer(Trainer):
    """HuggingFace Trainer subclass that replaces CE with Focal Loss."""
    def __init__(self, *args, focal_alpha=None, focal_gamma: float = 2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels").long()
        outputs = model(**inputs)
        loss    = self.focal_loss(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
def compute_class_weights(tokenized_dataset) -> torch.Tensor:
    """Inverse-frequency class weights from the training split (full, no sqrt softening)."""
    labels  = np.array(tokenized_dataset["train"]["labels"])
    counts  = np.bincount(labels, minlength=2)
    weights = len(labels) / (2.0 * counts)  # full inverse-frequency weighting
    return torch.tensor(weights.astype("float32"), dtype=torch.float32)


def default_training_args(output_dir: str) -> TrainingArguments:
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=1e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        remove_unused_columns=False,
    )


def find_best_threshold(trainer, tokenized_dataset) -> float:
    """Threshold that maximises minority-class F1 on the validation set."""
    pred      = trainer.predict(tokenized_dataset["validation"])
    probs1    = torch.softmax(torch.tensor(pred.predictions), dim=1)[:, 1].numpy()
    prec, rec, th = precision_recall_curve(pred.label_ids, probs1)
    f1s       = [2*p*r/(p+r) if (p+r) > 0 else 0.0 for p, r in zip(prec[:-1], rec[:-1])]
    return float(th[int(np.argmax(f1s))])


def evaluate_on_test(trainer, tokenized_dataset, threshold: float, save_path: str):
    """Print full test-set metrics and save ROC data to .npz for later plotting."""
    pred   = trainer.predict(tokenized_dataset["test"])
    probs1 = torch.softmax(torch.tensor(pred.predictions), dim=1)[:, 1].numpy()
    y_true = pred.label_ids
    y_hat  = (probs1 >= threshold).astype(int)

    print("Confusion matrix (TEST):\n", confusion_matrix(y_true, y_hat))
    print(classification_report(y_true, y_hat, digits=3))
    print("ROC-AUC:", roc_auc_score(y_true, probs1))
    print("PR-AUC: ", average_precision_score(y_true, probs1))
    print("Threshold used:", threshold)

    np.savez(save_path, y_true=y_true.astype(int), y_prob1=probs1.astype(float))
    print(f"Saved: {save_path}")
    return probs1, y_true


def run_experiment(model_ckpt, text_col, output_dir, roc_save_path,
                   max_length=128, seed=42):
    """
    Full train → threshold-select → evaluate pipeline for one model/text-column combo.

    Returns: (trainer, tokenized_dataset, best_threshold)
    """
    set_seed(seed)
    tokenized, raw, tok = build_tokenized_dataset(model_ckpt, text_col, max_length)
    collator            = DataCollatorWithPadding(tokenizer=tok)
    alpha               = compute_class_weights(tokenized)

    model   = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=2)
    args    = default_training_args(output_dir)

    trainer = FocalTrainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        data_collator=collator,
        compute_metrics=compute_metrics,
        focal_alpha=alpha,
        focal_gamma=2.0,
    )
    trainer.train()

    best_thresh = find_best_threshold(trainer, tokenized)
    print(f"\nBest threshold (val): {best_thresh:.4f}")
    evaluate_on_test(trainer, tokenized, best_thresh, roc_save_path)

    return trainer, tokenized, raw, tok, best_thresh

## 5. DistilBERT — Prognosis Text

Input: translated prognosis note only (`prognosis_english`), `max_length=64`.

In [ ]:
trainer_db, tokenized_db, raw_db, tok_db, thresh_db = run_experiment(
    model_ckpt   = "distilbert/distilbert-base-uncased",
    text_col     = TEXT_COL_EN,
    output_dir   = "results/distilbert_prognosis",
    roc_save_path= "rocdata_distilbert_prognosis_test.npz",
    max_length   = 64,
)

## 6. RoBERTa — Prognosis Text

Input: translated prognosis note only (`prognosis_english`), `max_length=128`.

In [ ]:
trainer_rb, tokenized_rb, raw_rb, tok_rb, thresh_rb = run_experiment(
    model_ckpt   = "roberta-base",
    text_col     = TEXT_COL_EN,
    output_dir   = "results/roberta_prognosis",
    roc_save_path= "rocdata_roberta_prognosis_test.npz",
    max_length   = 128,
)

## 7. DistilBERT — All Texts

Input: prognosis note + signs & symptoms concatenated (`all_texts`), `max_length=128`.

In [ ]:
trainer_db_all, tokenized_db_all, raw_db_all, tok_db_all, thresh_db_all = run_experiment(
    model_ckpt   = "distilbert/distilbert-base-uncased",
    text_col     = "all_texts",
    output_dir   = "results/distilbert_alltexts",
    roc_save_path= "rocdata_distilbert_alltexts_test.npz",
    max_length   = 128,
)

## 8. RoBERTa — All Texts

Input: prognosis note + signs & symptoms concatenated (`all_texts`), `max_length=128`.

In [ ]:
trainer_rb_all, tokenized_rb_all, raw_rb_all, tok_rb_all, thresh_rb_all = run_experiment(
    model_ckpt   = "roberta-base",
    text_col     = "all_texts",
    output_dir   = "results/roberta_alltexts",
    roc_save_path= "rocdata_roberta_alltexts_test.npz",
    max_length   = 128,
)

## 9. Danish-BERT — Untranslated Danish Text

Input: original untranslated Danish clinical text (`prognosis_danish`), `max_length=128`.

`Maltehb/danish-bert-botxo` is a BERT-base model pre-trained on Danish text. Using the untranslated source avoids any translation artefacts introduced by the English pipeline.

In [ ]:
trainer_da, tokenized_da, raw_da, tok_da, thresh_da = run_experiment(
    model_ckpt   = "Maltehb/danish-bert-botxo",
    text_col     = TEXT_COL_DANISH,
    output_dir   = "results/danishbert_alltexts",
    roc_save_path= "rocdata_danishbert_alltexts_test.npz",
    max_length   = 128,
)

## 10. ROC Curve Visualisation

All five models overlaid on a single publication-quality figure.
Edit `MODEL_FILES` to add or remove entries.

In [ ]:
MODEL_FILES = [
    ("DistilBERT (Prognosis)",   "rocdata_distilbert_prognosis_test.npz",  "#66c2a5"),
    ("RoBERTa (Prognosis)",      "rocdata_roberta_prognosis_test.npz",     "#fc8d62"),
    ("DistilBERT (All texts)",   "rocdata_distilbert_alltexts_test.npz",   "#8da0cb"),
    ("RoBERTa (All texts)",      "rocdata_roberta_alltexts_test.npz",      "#e78ac3"),
    ("Danish-BERT (All texts)",  "rocdata_danishbert_alltexts_test.npz",   "#a6d854"),
]

fig, ax = plt.subplots(figsize=(7, 6), dpi=150)

for label, path, color in MODEL_FILES:
    if not os.path.exists(path):
        print(f"Skipping {label}: {path} not found")
        continue
    d   = np.load(path)
    fpr, tpr, _ = roc_curve(d["y_true"], d["y_prob1"])
    auc = roc_auc_score(d["y_true"], d["y_prob1"])
    ax.plot(fpr, tpr, linewidth=2.5, color=color, label=f"{label}  AUC={auc:.2f}")

ax.plot([0, 1], [0, 1], "--", color="black", linewidth=1.5)
ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate",
       title="ROC Curves – Test Set", xlim=(0, 1), ylim=(0, 1))
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="lower right", frameon=False)
plt.tight_layout()
plt.savefig("roc_transformers_comparison.png", dpi=600, bbox_inches="tight")
plt.show()

## 11. Bootstrap Confidence Intervals

2 000-sample bootstrap for ROC-AUC, PR-AUC, macro-F1, and class-1 precision/recall.
Update `thresholds` with the `thresh_*` values printed during training.

In [ ]:
def bootstrap_ci(y_true, y_prob, threshold=0.5, n_boot=2000, seed=42):
    rng     = np.random.default_rng(seed)
    n       = len(y_true)
    metrics = {k: [] for k in ["roc_auc", "pr_auc", "f1_macro", "prec_c1", "rec_c1"]}

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt, yp = y_true[idx], y_prob[idx]
        if len(np.unique(yt)) < 2:
            continue
        yh = (yp >= threshold).astype(int)
        metrics["roc_auc"].append(roc_auc_score(yt, yp))
        metrics["pr_auc"].append(average_precision_score(yt, yp))
        metrics["f1_macro"].append(precision_recall_fscore_support(yt, yh, average="macro")[2])
        p1 = precision_recall_fscore_support(yt, yh, average=None, labels=[0, 1])[0][1]
        r1 = precision_recall_fscore_support(yt, yh, average=None, labels=[0, 1])[1][1]
        metrics["prec_c1"].append(p1)
        metrics["rec_c1"].append(r1)

    def ci(vals):
        v = np.asarray(vals)
        return np.mean(v), np.percentile(v, 2.5), np.percentile(v, 97.5)

    return {k: ci(v) for k, v in metrics.items()}, len(metrics["roc_auc"])


# Update thresholds with values printed after each experiment
MODELS_CI = [
    ("DistilBERT (Prognosis)",  "rocdata_distilbert_prognosis_test.npz",  thresh_db),
    ("RoBERTa (Prognosis)",     "rocdata_roberta_prognosis_test.npz",     thresh_rb),
    ("DistilBERT (All texts)",  "rocdata_distilbert_alltexts_test.npz",   thresh_db_all),
    ("RoBERTa (All texts)",     "rocdata_roberta_alltexts_test.npz",      thresh_rb_all),
    ("Danish-BERT (All texts)", "rocdata_danishbert_alltexts_test.npz",   thresh_da),
]

for name, path, thr in MODELS_CI:
    if not os.path.exists(path):
        print(f"Skipping {name}"); continue
    d = np.load(path)
    stats, n_used = bootstrap_ci(d["y_true"].astype(int), d["y_prob1"].astype(float),
                                  threshold=thr)
    print(f"\n{name}  (threshold={thr:.4f}, n_boot={n_used})")
    for metric, (mean, lo, hi) in stats.items():
        print(f"  {metric:15s}: {mean:.3f}  ({lo:.3f}–{hi:.3f})")